# v7d — DeBERTa / SciBERT Fine-Tuning

## Overview

This notebook fine-tunes pre-trained transformer models on our EOS vs Non-EOS classification task.
Unlike TF-IDF (bag-of-words), transformers understand word ORDER, context, and deep semantics.

## Why Transformers?

| Feature | TF-IDF + RF (v6 best) | Transformer (fine-tuned) |
|---------|----------------------|-------------------------|
| Input representation | Sparse bag-of-words | Contextual embeddings |
| Word order | Ignored | Preserved |
| Semantics | Token frequency only | Full contextual meaning |
| Vocab | Fixed at training | Sub-word tokenizer (handles OOV) |
| Mixed language | Separate word/char n-grams | Multilingual pre-training |
| Training time | ~30 seconds | ~30-60 minutes (GPU) / 4-8 hours (CPU) |
| Inference time | <1ms | ~50-200ms |

## Models

### 1. DeBERTa-v3-base
- **Paper**: DeBERTa: Decoding-enhanced BERT with Disentangled Attention (Microsoft, 2021)
- **Why**: State-of-the-art on many NLU benchmarks, disentangled attention handles position
  and content separately, better at capturing token relationships
- **Size**: 86M parameters, 768-dim hidden, 12 layers
- **Tokenizer**: SentencePiece (handles sub-words, good for technical vocab)

### 2. SciBERT
- **Paper**: SciBERT: A Pretrained Language Model for Scientific Text (Allen AI, 2019)
- **Why**: Pre-trained on 1.14M scientific papers (CS + biomedical). Semiconductor FA
  descriptions contain lots of technical/scientific vocabulary that SciBERT was exposed to.
- **Size**: 110M parameters, 768-dim hidden, 12 layers
- **Tokenizer**: WordPiece (similar to BERT)

## Expected Runtime

| Hardware | DeBERTa (3 epochs) | SciBERT (3 epochs) |
|----------|-------------------|-------------------|
| GPU (T4/A100) | 10-20 min | 8-15 min |
| CPU (14 cores) | 4-8 hours | 3-6 hours |

⚠️ **If you only have CPU**: Consider running overnight or on a GPU-enabled machine.
The notebook saves checkpoints every epoch so you can resume.

## Kernel
`efaai_v3` (Python 3.12) — needs: torch, transformers, sentence-transformers

In [1]:
# §0 — Imports & Hardware Detection
import os, json, warnings, time, sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, precision_score, recall_score, accuracy_score,
    classification_report
)

from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)

warnings.filterwarnings('ignore')
np.random.seed(42)
torch.manual_seed(42)

# Hardware detection
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'PyTorch version: {torch.__version__}')
print(f'Device: {device}')
if device.type == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'Memory: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')
else:
    print(f'⚠️  Running on CPU — expect 4-8 hours for full training')
    print(f'    Consider: reducing MAX_EPOCHS to 1, or using a GPU machine')

ROOT = os.path.abspath(os.getcwd())
if not os.path.exists(os.path.join(ROOT, 'data')):
    ROOT = os.path.abspath(os.path.join(ROOT, '..'))

DATA_DIR    = os.path.join(ROOT, 'data')
MODELS_DIR  = os.path.join(ROOT, 'models', 'v7d')
RESULTS_DIR = os.path.join(ROOT, 'results')
os.makedirs(MODELS_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

CSV_FILE    = os.path.join(DATA_DIR, 'v5a_eos_vs_noneos.csv')
TEXT_COL    = 'PSI Failure Desc'
LABEL_COL   = 'label'
RANDOM_STATE = 42

print(f'\nROOT: {ROOT}')

PyTorch version: 2.10.0+cpu
Device: cpu
⚠️  Running on CPU — expect 4-8 hours for full training
    Consider: reducing MAX_EPOCHS to 1, or using a GPU machine

ROOT: <project-root>


## §1 — Configuration

Adjust these parameters based on your hardware:
- **GPU**: Use defaults (batch_size=16, epochs=3)
- **CPU**: Reduce to batch_size=8, epochs=1-2, or set `MAX_LEN=128`

In [2]:
# ══════════════════════════════════════════════════════════════════
#  CONFIGURATION — Adjust for your hardware
# ══════════════════════════════════════════════════════════════════

# Model to fine-tune (change to switch between DeBERTa and SciBERT)
MODEL_NAME = 'microsoft/deberta-v3-base'   # Option 1: DeBERTa
# MODEL_NAME = 'allenai/scibert_scivocab_uncased'  # Option 2: SciBERT

# Training parameters
MAX_LEN       = 256      # Max token length (most FA texts are <100 tokens)
BATCH_SIZE    = 16       # Reduce to 8 on CPU with limited RAM
MAX_EPOCHS    = 3        # 3 epochs usually sufficient; reduce to 1 for quick test
LEARNING_RATE = 2e-5     # Standard for transformer fine-tuning
WARMUP_RATIO  = 0.1      # 10% warmup steps
WEIGHT_DECAY  = 0.01     # L2 regularisation

# Early stopping
PATIENCE = 2  # Stop if validation F1 doesn't improve for 2 epochs

# Class weights (EOS is minority ~25.5%)
# Higher weight for EOS forces the model to focus on the minority class
USE_CLASS_WEIGHTS = True

print(f'Model: {MODEL_NAME}')
print(f'Max length: {MAX_LEN}')
print(f'Batch size: {BATCH_SIZE}')
print(f'Epochs: {MAX_EPOCHS}')
print(f'Learning rate: {LEARNING_RATE}')
print(f'Device: {device}')

Model: microsoft/deberta-v3-base
Max length: 256
Batch size: 16
Epochs: 3
Learning rate: 2e-05
Device: cpu


In [3]:
# §2 — Load Data & Split
df = pd.read_csv(CSV_FILE)
df[TEXT_COL] = df[TEXT_COL].astype(str).str.strip()
df = df[df[TEXT_COL].str.len() > 3].reset_index(drop=True)
print(f'Loaded: {df.shape}')

le = LabelEncoder()
df['y'] = le.fit_transform(df[LABEL_COL])
EOS_IDX = list(le.classes_).index('EOS')
NUM_LABELS = len(le.classes_)
print(f'Classes: {list(le.classes_)}, EOS index: {EOS_IDX}, Num labels: {NUM_LABELS}')

X_text = df[TEXT_COL].values
y = df['y'].values

# Same split as v5a/v6 for fair comparison
X_train_text, X_test_text, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)

# Further split train into train+val for early stopping
X_tr_text, X_val_text, y_tr, y_val = train_test_split(
    X_train_text, y_train, test_size=0.1, stratify=y_train, random_state=RANDOM_STATE
)

print(f'Train: {len(y_tr)}, Val: {len(y_val)}, Test: {len(y_test)}')
print(f'Train EOS%: {(y_tr==EOS_IDX).mean()*100:.1f}%')

# Class weights
if USE_CLASS_WEIGHTS:
    from sklearn.utils.class_weight import compute_class_weight
    weights = compute_class_weight('balanced', classes=np.unique(y_tr), y=y_tr)
    class_weights = torch.FloatTensor(weights).to(device)
    print(f'Class weights: {weights}')
else:
    class_weights = None

Loaded: (13910, 3)
Classes: ['EOS', 'Non-EOS'], EOS index: 0, Num labels: 2
Train: 10015, Val: 1113, Test: 2782
Train EOS%: 25.5%
Class weights: [1.95988258 0.67124665]


## §3 — Tokenizer & Dataset

The tokenizer converts text to token IDs that the transformer understands.
- DeBERTa uses SentencePiece: splits words into sub-word units
- SciBERT uses WordPiece: similar but was trained on scientific text

Example: `"overvoltage damage"` → `[101, 1952, 7045, 3305, 4783, 102]`

We pad/truncate all texts to `MAX_LEN` tokens for batching.

In [4]:
# §3 — Load tokenizer
print(f'Loading tokenizer: {MODEL_NAME} ...')
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
print(f'Tokenizer loaded. Vocab size: {tokenizer.vocab_size}')

# Check token length distribution
sample_lengths = [len(tokenizer.encode(t, add_special_tokens=True)) for t in X_tr_text[:500]]
print(f'\nToken length stats (sample of 500):')
print(f'  Mean: {np.mean(sample_lengths):.0f}')
print(f'  Median: {np.median(sample_lengths):.0f}')
print(f'  95th percentile: {np.percentile(sample_lengths, 95):.0f}')
print(f'  Max: {max(sample_lengths)}')
print(f'  MAX_LEN setting: {MAX_LEN} (texts longer will be truncated)')

class FADataset(Dataset):
    """PyTorch Dataset for FA failure descriptions."""
    def __init__(self, texts, labels, tokenizer, max_len):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len
    
    def __len__(self):
        return len(self.texts)
    
    def __getitem__(self, idx):
        text = str(self.texts[idx])
        encoding = self.tokenizer(
            text, max_length=self.max_len, padding='max_length',
            truncation=True, return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'labels': torch.tensor(self.labels[idx], dtype=torch.long)
        }

# Create datasets
train_dataset = FADataset(X_tr_text, y_tr, tokenizer, MAX_LEN)
val_dataset = FADataset(X_val_text, y_val, tokenizer, MAX_LEN)
test_dataset = FADataset(X_test_text, y_test, tokenizer, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE*2)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE*2)

print(f'\nDatasets created:')
print(f'  Train: {len(train_dataset)} samples, {len(train_loader)} batches')
print(f'  Val:   {len(val_dataset)} samples, {len(val_loader)} batches')
print(f'  Test:  {len(test_dataset)} samples, {len(test_loader)} batches')

Loading tokenizer: microsoft/deberta-v3-base ...


Tokenizer loaded. Vocab size: 128000

Token length stats (sample of 500):
  Mean: 9
  Median: 8
  95th percentile: 20
  Max: 33
  MAX_LEN setting: 256 (texts longer will be truncated)

Datasets created:
  Train: 10015 samples, 626 batches
  Val:   1113 samples, 35 batches
  Test:  2782 samples, 87 batches


## §4 — Model Loading

We load the pre-trained model and add a classification head on top.
The model architecture is:
```
Text → Tokenizer → [CLS] token embedding → 12 transformer layers → 
Linear(768 → 2) → Softmax → EOS/Non-EOS probability
```

Fine-tuning updates ALL weights (not just the classification head) but with a
very small learning rate (2e-5) to preserve pre-trained knowledge.

In [5]:
# §4 — Load pre-trained model
print(f'Loading model: {MODEL_NAME} ...')
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=NUM_LABELS
)
model = model.to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters: {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')
print(f'Model size: ~{total_params * 4 / 1e6:.0f} MB (float32)')

# Optimizer and scheduler
total_steps = len(train_loader) * MAX_EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

optimizer = AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = get_linear_schedule_with_warmup(
    optimizer, num_warmup_steps=warmup_steps, num_training_steps=total_steps
)

print(f'\nTraining plan:')
print(f'  Total steps: {total_steps}')
print(f'  Warmup steps: {warmup_steps}')
print(f'  Estimated time per epoch: {len(train_loader) * 0.5:.0f}s (GPU) / {len(train_loader) * 8:.0f}s (CPU)')

Loading model: microsoft/deberta-v3-base ...


Loading weights:   0%|          | 0/198 [00:00<?, ?it/s]

DebertaV2ForSequenceClassification LOAD REPORT from: microsoft/deberta-v3-base
Key                                     | Status     | 
----------------------------------------+------------+-
mask_predictions.LayerNorm.weight       | UNEXPECTED | 
mask_predictions.classifier.bias        | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.bias   | UNEXPECTED | 
lm_predictions.lm_head.bias             | UNEXPECTED | 
mask_predictions.dense.bias             | UNEXPECTED | 
mask_predictions.LayerNorm.bias         | UNEXPECTED | 
lm_predictions.lm_head.LayerNorm.weight | UNEXPECTED | 
mask_predictions.classifier.weight      | UNEXPECTED | 
lm_predictions.lm_head.dense.bias       | UNEXPECTED | 
lm_predictions.lm_head.dense.weight     | UNEXPECTED | 
mask_predictions.dense.weight           | UNEXPECTED | 
pooler.dense.bias                       | MISSING    | 
classifier.bias                         | MISSING    | 
classifier.weight                       | MISSING    | 
pooler.dense.weight      

Total parameters: 184,423,682
Trainable parameters: 184,423,682
Model size: ~738 MB (float32)

Training plan:
  Total steps: 1878
  Warmup steps: 187
  Estimated time per epoch: 313s (GPU) / 5008s (CPU)


## §5 — Training Loop

The training loop:
1. **Forward pass**: Text → model → loss (cross-entropy with class weights)
2. **Backward pass**: Compute gradients
3. **Update**: Adjust all model weights by tiny amounts (lr=2e-5)
4. **Validate**: After each epoch, check Macro-F1 on validation set
5. **Early stopping**: If val Macro-F1 doesn't improve for `PATIENCE` epochs, stop

**Progress indicator**: Shows epoch, batch, loss, and estimated time remaining.

In [ ]:
# §5 — Training loop with early stopping
from torch.nn import CrossEntropyLoss

loss_fn = CrossEntropyLoss(weight=class_weights) if class_weights is not None else CrossEntropyLoss()

best_val_f1 = 0
patience_counter = 0
history = []

print(f'\n{"="*70}')
print(f'  TRAINING: {MODEL_NAME}')
print(f'{"="*70}')

for epoch in range(MAX_EPOCHS):
    # ── Train ──────────────────────────────────────────────────────────
    model.train()
    train_loss = 0
    train_preds = []
    train_labels = []
    t0_epoch = time.time()
    
    for batch_idx, batch in enumerate(train_loader):
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)
        
        optimizer.zero_grad()
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        # Cast logits to float32 — DeBERTa may output float16
        loss = loss_fn(outputs.logits.float(), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        
        train_loss += loss.item()
        preds = outputs.logits.argmax(dim=1).cpu().numpy()
        train_preds.extend(preds)
        train_labels.extend(labels.cpu().numpy())
        
        # Progress every 50 batches
        if (batch_idx + 1) % 50 == 0 or batch_idx == len(train_loader) - 1:
            elapsed = time.time() - t0_epoch
            eta = elapsed / (batch_idx + 1) * (len(train_loader) - batch_idx - 1)
            print(f'  Epoch {epoch+1}/{MAX_EPOCHS} | Batch {batch_idx+1}/{len(train_loader)} | '
                  f'Loss: {train_loss/(batch_idx+1):.4f} | ETA: {eta:.0f}s')
    
    train_f1 = f1_score(train_labels, train_preds, average='macro')
    epoch_time = time.time() - t0_epoch
    
    # ── Validate ───────────────────────────────────────────────────────
    model.eval()
    val_preds = []
    val_labels_list = []
    
    with torch.no_grad():
        for batch in val_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids=input_ids, attention_mask=attention_mask)
            preds = outputs.logits.argmax(dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_labels_list.extend(batch['labels'].numpy())
    
    val_f1 = f1_score(val_labels_list, val_preds, average='macro')
    val_acc = accuracy_score(val_labels_list, val_preds)
    
    history.append({
        'epoch': epoch + 1,
        'train_loss': train_loss / len(train_loader),
        'train_f1': train_f1,
        'val_f1': val_f1,
        'val_acc': val_acc,
        'time_s': epoch_time,
    })
    
    print(f'\n  Epoch {epoch+1} Summary:')
    print(f'    Train Loss: {train_loss/len(train_loader):.4f} | Train F1: {train_f1:.4f}')
    print(f'    Val F1: {val_f1:.4f} | Val Acc: {val_acc:.4f}')
    print(f'    Time: {epoch_time:.0f}s')
    
    # Early stopping
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        patience_counter = 0
        # Save best model
        torch.save(model.state_dict(), os.path.join(MODELS_DIR, 'best_transformer.pt'))
        print(f'    ✅ New best! Saved checkpoint.')
    else:
        patience_counter += 1
        print(f'    ⚠️  No improvement ({patience_counter}/{PATIENCE})')
        if patience_counter >= PATIENCE:
            print(f'    🛑 Early stopping triggered.')
            break

print(f'\nTraining complete. Best val Macro-F1: {best_val_f1:.4f}')


  TRAINING: microsoft/deberta-v3-base


## §6 — Evaluation on Test Set

Load the best checkpoint (highest validation F1) and evaluate on the held-out test set.
This gives the SAME test set as v5a/v6 for direct comparison.

In [ ]:
# §6 — Load best model and evaluate on test set
print('Loading best checkpoint...')
model.load_state_dict(torch.load(os.path.join(MODELS_DIR, 'best_transformer.pt'),
                                  map_location=device))
model.eval()

test_preds = []
test_probs = []
test_labels_list = []

with torch.no_grad():
    for batch in test_loader:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        outputs = model(input_ids=input_ids, attention_mask=attention_mask)
        # Cast to float32 for softmax (DeBERTa may output float16)
        logits = outputs.logits.float()
        probs = torch.softmax(logits, dim=1).cpu().numpy()
        preds = logits.argmax(dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_probs.extend(probs)
        test_labels_list.extend(batch['labels'].numpy())

test_preds = np.array(test_preds)
test_labels_arr = np.array(test_labels_list)

# Metrics
mf1 = f1_score(test_labels_arr, test_preds, average='macro')
acc = accuracy_score(test_labels_arr, test_preds)
ep = precision_score(test_labels_arr, test_preds, pos_label=EOS_IDX, zero_division=0)
er = recall_score(test_labels_arr, test_preds, pos_label=EOS_IDX, zero_division=0)
ef1 = f1_score(test_labels_arr, test_preds, pos_label=EOS_IDX, zero_division=0)

print(f'\n{"="*70}')
print(f'  TEST SET RESULTS: {MODEL_NAME}')
print(f'{"="*70}')
print(f'  Macro-F1:      {mf1:.4f}')
print(f'  Accuracy:      {acc:.4f}')
print(f'  EOS-Precision: {ep:.4f}')
print(f'  EOS-Recall:    {er:.4f}')
print(f'  EOS-F1:        {ef1:.4f}')
print(f'\n  vs v6 best (word+char TF-IDF + RF-SMOTE):')
print(f'  v6 Macro-F1: 0.7705')
print(f'  Delta:       {mf1 - 0.7705:+.4f}')
print(f'\n{classification_report(test_labels_arr, test_preds, target_names=le.classes_)}')

# Save results
results = {
    'model': MODEL_NAME,
    'macro_f1': float(mf1),
    'accuracy': float(acc),
    'eos_precision': float(ep),
    'eos_recall': float(er),
    'eos_f1': float(ef1),
    'epochs_trained': len(history),
    'best_val_f1': float(best_val_f1),
    'training_history': history,
    'device': str(device),
}

with open(os.path.join(RESULTS_DIR, 'v7d_transformer_results.json'), 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nSaved: results/v7d_transformer_results.json')

## §7 — Instructions for Second Model

To fine-tune SciBERT instead of DeBERTa:
1. In §1, change `MODEL_NAME` to `'allenai/scibert_scivocab_uncased'`
2. Restart kernel
3. Run all cells

Or — to compare both without re-running:
- Save this notebook as `v7d_deberta.ipynb`
- Copy and rename to `v7d_scibert.ipynb`
- Change MODEL_NAME and run

## Expected Results (from v5a experiments)

In v5a, SciBERT achieved ~0.74 Macro-F1 with 3 epochs on CPU (but may have been
under-trained). With proper class weights and more patient training, transformers
should reach 0.77-0.80 if the data has enough semantic signal beyond bag-of-words.